# 03 — Drivers da Mortalidade: Pacientes Mais Graves ou Cuidado Pior?

Decomposição do aumento de mortalidade do J96 usando três eixos independentes:
1. **Padronização direta** (Kitagawa) — separa efeito composição (demografia) do efeito taxa (sistema)
2. **Análise dentro-de-estrato** — mortalidade está subindo dentro das mesmas faixas etárias?
3. **Interação com infraestrutura** — o efeito sistema se concentra em hospitais sem UTI?

**Pergunta de Pesquisa:** RQ2 — O que está impulsionando o aumento da mortalidade?

**Depende de:** `resp_failure_sih.parquet`, `hospital_tags.parquet`, `hospital_icu_beds.parquet`

In [1]:
import sys
sys.path.insert(0, ".")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
from shared import (
    load_resp_failure, load_hospital_tags, load_icu_beds,
    setup_plot_style, save_plot, save_metrics,
    COLORS, ERA_COLORS, ERA_LABELS, ERA_ORDER,
)

setup_plot_style()
resp = load_resp_failure()
resp = resp[resp["year"] >= 2016].copy()
tags = load_hospital_tags()
icu_beds = load_icu_beds()

# Merge hospital metadata
resp = resp.merge(tags[["CNES", "icu_capacity_level", "broad_type", "nat_jur_category"]], on="CNES", how="left")
resp = resp.merge(icu_beds[["CNES", "icu_beds_sus"]], on="CNES", how="left")
resp["icu_beds_sus"] = resp["icu_beds_sus"].fillna(0)

resp["age_group"] = pd.cut(resp["age"], bins=[0, 1, 18, 40, 60, 75, 120],
                            labels=["<1", "1-17", "18-39", "40-59", "60-74", "75+"])
resp["has_icu_days"] = (resp["icu_days"] > 0).astype(int)

comorbidity_cols = [c for c in resp.columns if c.startswith("comorbidity_") and c != "comorbidity_count"]

# Define periods for comparison
pre_covid = resp[resp["covid_era"] == "pre_covid"]
post_late = resp[resp["covid_era"] == "post_covid_late"]

print(f"Total admissions: {len(resp):,}")
print(f"Pre-COVID: {len(pre_covid):,}  |  Post-COVID Late: {len(post_late):,}")
print(f"Years: {sorted(resp['year'].dropna().unique().astype(int))}")

Total admissions: 116,374
Pre-COVID: 44,447  |  Post-COVID Late: 28,357
Years: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


## 1. Evolução do Perfil dos Pacientes

Primeiro, verificar se os pacientes estão chegando mais graves ao longo do tempo. Se a composição demográfica mudou, parte do aumento de mortalidade é mecânica (efeito composição).

In [2]:
yearly_profile = resp.groupby("year").agg(
    n=("MORTE", "count"),
    mortality=("MORTE", "mean"),
    mean_age=("age", "mean"),
    pct_75plus=("age", lambda x: (x >= 75).mean()),
    pct_60plus=("age", lambda x: (x >= 60).mean()),
    pct_emergency=("is_emergency", "mean"),
    mean_comorbidity=("comorbidity_count", "mean"),
    pct_male=("is_male", "mean"),
    mean_los=("DIAS_PERM", "mean"),
    pct_no_icu_hosp=("icu_capacity_level", lambda x: (x.astype(str) == "no_icu").mean()),
).reset_index()

fig, axes = plt.subplots(2, 4, figsize=(22, 10))

metrics_plot = [
    ("mortality", "Taxa de Mortalidade", COLORS["danger"], True),
    ("mean_age", "Idade Média", COLORS["primary"], False),
    ("pct_60plus", "% Pacientes 60+", COLORS["secondary"], True),
    ("pct_emergency", "% Urgência", COLORS["warning"], True),
    ("mean_comorbidity", "Comorbidades (Média)", COLORS["danger"], False),
    ("pct_male", "% Masculino", COLORS["primary"], True),
    ("mean_los", "Permanência Média (dias)", COLORS["muted"], False),
    ("pct_no_icu_hosp", "% Atendidos em Hosp. Sem UTI", COLORS["danger"], True),
]

for ax, (col, title, color, is_pct) in zip(axes.flat, metrics_plot):
    values = yearly_profile[col] * 100 if is_pct else yearly_profile[col]
    ax.plot(yearly_profile["year"], values, marker="o", linewidth=2.5, color=color)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xlabel("")
    ax.axvspan(2020, 2021.5, alpha=0.08, color="red")
    if is_pct:
        ax.yaxis.set_major_formatter(mtick.PercentFormatter())

fig.suptitle("Evolução do Perfil dos Pacientes J96 (2016–2025)", fontsize=14, fontweight="bold")
plt.tight_layout()
save_plot(fig, "patient_profile_trend", prefix="03")

# Kendall tau for key variables
for col, label in [("mean_age", "Idade média"), ("pct_60plus", "% 60+"), ("mortality", "Mortalidade"),
                    ("pct_no_icu_hosp", "% sem UTI")]:
    tau, p = stats.kendalltau(yearly_profile["year"], yearly_profile[col])
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    print(f"  Kendall τ {label}: {tau:.3f} (p={p:.4f}) {sig}")

  Saved: 03_patient_profile_trend.png
  Kendall τ Idade média: 0.333 (p=0.2164) ns
  Kendall τ % 60+: 0.422 (p=0.1083) ns
  Kendall τ Mortalidade: 0.556 (p=0.0286) *
  Kendall τ % sem UTI: 0.067 (p=0.8618) ns


## 2. Decomposição por Padronização Direta (Kitagawa)

Método: aplica as taxas de mortalidade por [faixa etária × sexo] do período pré-COVID (2016–2019) à composição demográfica de cada ano subsequente. A diferença entre a mortalidade **esperada** (se apenas a composição tivesse mudado) e a mortalidade **observada** revela o *efeito sistema*.

- **Efeito composição (demográfico)**: mortalidade esperada − mortalidade baseline
- **Efeito taxa (sistema)**: mortalidade observada − mortalidade esperada

Se o efeito sistema domina, o problema não é que os pacientes estão mais graves — é que o sistema está performando pior.

In [3]:
# Pre-COVID baseline rates by stratum
baseline_rates = pre_covid.groupby(["age_group", "is_male"], observed=True)["MORTE"].mean().to_dict()
baseline_overall = pre_covid["MORTE"].mean()

resp["expected_mort"] = resp.apply(
    lambda r: baseline_rates.get((r["age_group"], r["is_male"]), baseline_overall), axis=1
)

decomp = resp.groupby("year").agg(
    observed=("MORTE", "mean"),
    expected=("expected_mort", "mean"),
    n=("MORTE", "count"),
).reset_index()

decomp["composition_effect"] = decomp["expected"] - baseline_overall
decomp["rate_effect"] = decomp["observed"] - decomp["expected"]
decomp["total_change"] = decomp["observed"] - baseline_overall

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Panel A: observed vs expected
ax = axes[0]
ax.plot(decomp["year"], decomp["observed"] * 100, marker="o", linewidth=2.5,
        color=COLORS["danger"], label="Observada", zorder=5)
ax.plot(decomp["year"], decomp["expected"] * 100, marker="s", linewidth=2,
        color=COLORS["muted"], linestyle="--", label="Esperada (demografia pré-COVID)")
ax.fill_between(decomp["year"], decomp["expected"] * 100, decomp["observed"] * 100,
                where=decomp["observed"] > decomp["expected"],
                alpha=0.15, color=COLORS["danger"], label="Efeito taxa (piora do sistema)")
ax.fill_between(decomp["year"], decomp["expected"] * 100, decomp["observed"] * 100,
                where=decomp["observed"] <= decomp["expected"],
                alpha=0.15, color=COLORS["success"], label="Efeito taxa (melhora do sistema)")
ax.axhline(baseline_overall * 100, color="black", linestyle=":", alpha=0.3, label=f"Baseline pré-COVID ({baseline_overall:.1%})")
ax.axvspan(2020, 2021.5, alpha=0.06, color="red")
ax.set_title("A. Mortalidade Observada vs Esperada", fontweight="bold")
ax.set_ylabel("Mortalidade (%)")
ax.legend(fontsize=8, loc="upper left")

# Panel B: stacked decomposition
ax = axes[1]
width = 0.35
years = decomp["year"].values
ax.bar(years - width/2, decomp["composition_effect"] * 100, width, label="Efeito composição (demografia)",
       color=COLORS["primary"], alpha=0.8)
ax.bar(years + width/2, decomp["rate_effect"] * 100, width, label="Efeito taxa (sistema)",
       color=COLORS["danger"], alpha=0.8)
ax.axhline(0, color="black", linewidth=0.5)
ax.axvspan(2020, 2021.5, alpha=0.06, color="red")
ax.set_title("B. Decomposição Anual (pp vs baseline pré-COVID)", fontweight="bold")
ax.set_ylabel("Variação (pp)")
ax.legend(fontsize=8)

# Panel C: pie for 2024
ax = axes[2]
row_2024 = decomp[decomp["year"] == 2024].iloc[0]
total_abs = abs(row_2024["composition_effect"]) + abs(row_2024["rate_effect"])
pct_comp = abs(row_2024["composition_effect"]) / total_abs * 100
pct_rate = abs(row_2024["rate_effect"]) / total_abs * 100
wedges, texts, autotexts = ax.pie(
    [pct_comp, pct_rate],
    labels=[f"Demografia\n({row_2024['composition_effect']*100:+.1f}pp)",
            f"Sistema\n({row_2024['rate_effect']*100:+.1f}pp)"],
    colors=[COLORS["primary"], COLORS["danger"]],
    autopct="%1.0f%%", startangle=90, textprops={"fontsize": 11}
)
ax.set_title(f"C. Decomposição 2024 (total: {row_2024['total_change']*100:+.1f}pp)", fontweight="bold")

fig.suptitle("Decomposição Kitagawa do Aumento de Mortalidade J96", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
save_plot(fig, "mortality_decomposition", prefix="03")

print("\nDecomposição por ano (ref: mortalidade pré-COVID = {:.1%}):".format(baseline_overall))
print(f"{'Ano':>6} {'Observ.':>8} {'Esper.':>8} {'Composição':>12} {'Taxa':>10} {'Total':>8}")
for _, r in decomp.iterrows():
    print(f"{int(r['year']):>6} {r['observed']:>7.1%} {r['expected']:>7.1%} "
          f"{r['composition_effect']*100:>+10.1f}pp {r['rate_effect']*100:>+8.1f}pp {r['total_change']*100:>+6.1f}pp")

  Saved: 03_mortality_decomposition.png

Decomposição por ano (ref: mortalidade pré-COVID = 31.1%):
   Ano  Observ.   Esper.   Composição       Taxa    Total
  2016   31.5%   31.4%       +0.3pp     +0.1pp   +0.3pp
  2017   31.0%   30.7%       -0.5pp     +0.3pp   -0.1pp
  2018   30.0%   30.5%       -0.6pp     -0.5pp   -1.1pp
  2019   31.4%   31.5%       +0.4pp     -0.1pp   +0.3pp
  2020   33.7%   39.8%       +8.6pp     -6.1pp   +2.5pp
  2021   31.3%   36.3%       +5.2pp     -5.0pp   +0.2pp
  2022   34.4%   32.8%       +1.6pp     +1.6pp   +3.2pp
  2023   33.3%   30.9%       -0.3pp     +2.4pp   +2.2pp
  2024   37.2%   33.3%       +2.2pp     +3.9pp   +6.1pp
  2025   35.6%   33.5%       +2.4pp     +2.1pp   +4.4pp


## 3. Análise Dentro-de-Estrato: Mortalidade Subiu em TODAS as Faixas Etárias

Se a mortalidade sobe uniformemente dentro de cada faixa etária, não é um efeito de composição — é uma piora sistêmica que atinge todos os grupos.

In [4]:
age_year = resp.groupby(["year", "age_group"], observed=True).agg(
    n=("MORTE", "count"),
    deaths=("MORTE", "sum"),
    mortality=("MORTE", "mean"),
).reset_index()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# Panel A: mortality trends by age group
age_palette = ["#06B6D4", "#8B5CF6", "#F59E0B", "#EF4444", "#10B981", "#3B82F6"]
for (ag, grp), color in zip(age_year.groupby("age_group", observed=True), age_palette):
    grp = grp.sort_values("year")
    ax1.plot(grp["year"], grp["mortality"] * 100, marker="o", linewidth=2.5, label=str(ag), color=color)

ax1.axvspan(2020, 2021.5, alpha=0.06, color="red")
ax1.set_title("A. Mortalidade por Faixa Etária (2016–2025)", fontweight="bold")
ax1.set_xlabel("Ano")
ax1.set_ylabel("Mortalidade (%)")
ax1.legend(title="Faixa Etária", fontsize=9)

# Panel B: pre vs post-COVID late change by age group
change_data = []
for ag in ["<1", "1-17", "18-39", "40-59", "60-74", "75+"]:
    m_pre = pre_covid[pre_covid["age_group"] == ag]["MORTE"].mean()
    m_post = post_late[post_late["age_group"] == ag]["MORTE"].mean()
    n_pre = len(pre_covid[pre_covid["age_group"] == ag])
    n_post = len(post_late[post_late["age_group"] == ag])
    if n_pre > 50 and n_post > 50:
        # Chi-squared test for significance
        d_pre = pre_covid[pre_covid["age_group"] == ag]["MORTE"].sum()
        d_post = post_late[post_late["age_group"] == ag]["MORTE"].sum()
        table = [[d_pre, n_pre - d_pre], [d_post, n_post - d_post]]
        chi2, pval, _, _ = stats.chi2_contingency(table)
        change_data.append({"age_group": ag, "pre": m_pre, "post": m_post,
                            "change_pp": (m_post - m_pre) * 100, "n_pre": n_pre, "n_post": n_post,
                            "chi2": chi2, "p": pval})

change_df = pd.DataFrame(change_data)
bar_colors = [COLORS["danger"] if p < 0.05 else COLORS["muted"] for p in change_df["p"]]
bars = ax2.bar(change_df["age_group"], change_df["change_pp"], color=bar_colors, alpha=0.85)
for bar, row in zip(bars, change_df.itertuples()):
    sig = "***" if row.p < 0.001 else "**" if row.p < 0.01 else "*" if row.p < 0.05 else "ns"
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
             f"{row.change_pp:+.1f}pp\n{sig}", ha="center", fontsize=9, fontweight="bold")

ax2.axhline(0, color="black", linewidth=0.5)
ax2.set_title("B. Variação de Mortalidade: Pré-COVID → Pós-COVID Tardio", fontweight="bold")
ax2.set_xlabel("Faixa Etária")
ax2.set_ylabel("Variação (pp)")

fig.suptitle("Mortalidade por Faixa Etária — A Piora É Universal", fontsize=14, fontweight="bold")
plt.tight_layout()
save_plot(fig, "mortality_by_age_trend", prefix="03")

print("\nVariação dentro-de-estrato (pré-COVID → pós-COVID tardio):")
print(f"{'Faixa':>8} {'Pré-COVID':>10} {'Pós-Tardio':>12} {'Δ (pp)':>8} {'n pre':>8} {'n post':>8} {'χ²':>8} {'p-value':>10}")
for _, r in change_df.iterrows():
    sig = "***" if r['p'] < 0.001 else "**" if r['p'] < 0.01 else "*" if r['p'] < 0.05 else "ns"
    print(f"{r['age_group']:>8} {r['pre']:>9.1%} {r['post']:>11.1%} {r['change_pp']:>+7.1f} "
          f"{r['n_pre']:>7,} {r['n_post']:>7,} {r['chi2']:>7.1f} {r['p']:>9.4f} {sig}")

  Saved: 03_mortality_by_age_trend.png

Variação dentro-de-estrato (pré-COVID → pós-COVID tardio):
   Faixa  Pré-COVID   Pós-Tardio   Δ (pp)    n pre   n post       χ²    p-value
      <1      2.4%        3.6%    +1.1   4,049   2,332     6.2    0.0125 *
    1-17      3.3%        3.8%    +0.5  12,394   7,360     3.4    0.0649 ns
   18-39     22.4%       26.2%    +3.7   3,520   2,025     9.7    0.0019 **
   40-59     39.7%       43.5%    +3.7   7,282   4,163    15.0    0.0001 ***
   60-74     50.0%       54.3%    +4.3   9,371   6,551    28.8    0.0000 ***
     75+     64.3%       68.7%    +4.4   7,670   5,823    28.5    0.0000 ***


## 4. Interação com Infraestrutura de UTI

O efeito sistema se concentra em hospitais sem UTI? Se sim, a causa provável é falta de acesso a cuidado intensivo, não uma piora generalizada do cuidado.

Decomposição da mortalidade por nível de UTI e por acesso individual do paciente a dias de UTI.

In [5]:
# Mortality trend by ICU access (patient-level: has_icu_days)
icu_mort = []
for yr in sorted(resp["year"].dropna().unique().astype(int)):
    yr_data = resp[resp["year"] == yr]
    with_icu = yr_data[yr_data["has_icu_days"] == 1]
    no_icu = yr_data[yr_data["has_icu_days"] == 0]
    icu_mort.append({
        "year": yr,
        "pct_icu": len(with_icu) / len(yr_data) if len(yr_data) > 0 else 0,
        "mort_icu": with_icu["MORTE"].mean() if len(with_icu) > 0 else np.nan,
        "mort_no_icu": no_icu["MORTE"].mean() if len(no_icu) > 0 else np.nan,
        "n_icu": len(with_icu),
        "n_no_icu": len(no_icu),
    })
icu_df = pd.DataFrame(icu_mort)
icu_df["gap"] = icu_df["mort_no_icu"] - icu_df["mort_icu"]

# Also by hospital ICU capacity level
icu_levels = ["no_icu", "small_icu", "medium_icu", "large_icu"]
icu_level_labels = {"no_icu": "Sem UTI", "small_icu": "UTI Peq.", "medium_icu": "UTI Méd.", "large_icu": "UTI Grande"}
icu_level_colors = {"no_icu": COLORS["danger"], "small_icu": COLORS["warning"],
                    "medium_icu": COLORS["primary"], "large_icu": COLORS["success"]}

level_yr = resp.groupby(["year", "icu_capacity_level"], observed=True).agg(
    mortality=("MORTE", "mean"), n=("MORTE", "count")
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# Panel A: patient-level ICU access — mortality divergence
ax = axes[0]
ax.plot(icu_df["year"], icu_df["mort_icu"] * 100, marker="o", linewidth=2.5,
        color=COLORS["success"], label="Com dias de UTI")
ax.plot(icu_df["year"], icu_df["mort_no_icu"] * 100, marker="s", linewidth=2.5,
        color=COLORS["danger"], label="Sem dias de UTI")
ax.fill_between(icu_df["year"], icu_df["mort_icu"] * 100, icu_df["mort_no_icu"] * 100,
                alpha=0.1, color=COLORS["danger"])
ax.axvspan(2020, 2021.5, alpha=0.06, color="red")
ax.set_title("A. Divergência de Mortalidade: Com vs Sem UTI", fontweight="bold")
ax.set_ylabel("Mortalidade (%)")
ax.legend(fontsize=9)

# Panel B: gap widening
ax = axes[1]
bar_colors_gap = [COLORS["success"] if g < 5 else COLORS["warning"] if g < 10 else COLORS["danger"]
                  for g in icu_df["gap"] * 100]
ax.bar(icu_df["year"], icu_df["gap"] * 100, color=bar_colors_gap, alpha=0.8)
ax.axvspan(2020, 2021.5, alpha=0.06, color="red")
ax.set_title("B. Gap de Mortalidade (sem UTI − com UTI)", fontweight="bold")
ax.set_ylabel("Gap (pp)")
for i, (yr, gap) in enumerate(zip(icu_df["year"], icu_df["gap"] * 100)):
    ax.text(yr, gap + 0.3, f"{gap:.0f}", ha="center", fontsize=9, fontweight="bold")

# Panel C: hospital-level ICU capacity — mortality trends
ax = axes[2]
for lvl in icu_levels:
    grp = level_yr[level_yr["icu_capacity_level"] == lvl].sort_values("year")
    if len(grp) > 0:
        ax.plot(grp["year"], grp["mortality"] * 100, marker="o", linewidth=2.5,
                label=icu_level_labels.get(lvl, lvl), color=icu_level_colors[lvl])
ax.axvspan(2020, 2021.5, alpha=0.06, color="red")
ax.set_title("C. Mortalidade por Capacidade UTI do Hospital", fontweight="bold")
ax.set_ylabel("Mortalidade (%)")
ax.legend(fontsize=9)

fig.suptitle("O Efeito Sistema Concentra-se nos Pacientes e Hospitais Sem UTI",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
save_plot(fig, "icu_interaction", prefix="03")

print("\nMortalidade por acesso a UTI (paciente):")
print(f"{'Ano':>6} {'% UTI':>7} {'Mort UTI':>9} {'Mort s/UTI':>11} {'Gap':>6}")
for _, r in icu_df.iterrows():
    print(f"{int(r['year']):>6} {r['pct_icu']:>6.1%} {r['mort_icu']:>8.1%} {r['mort_no_icu']:>10.1%} {r['gap']*100:>5.1f}pp")

  Saved: 03_icu_interaction.png

Mortalidade por acesso a UTI (paciente):
   Ano   % UTI  Mort UTI  Mort s/UTI    Gap
  2016  32.5%    28.5%      32.9%   4.4pp
  2017  30.7%    27.8%      32.5%   4.7pp
  2018  31.1%    25.5%      32.1%   6.6pp
  2019  32.9%    27.2%      33.5%   6.3pp
  2020  29.1%    36.7%      32.4%  -4.3pp
  2021  29.0%    31.6%      31.2%  -0.4pp
  2022  34.6%    31.3%      36.0%   4.7pp
  2023  36.2%    29.1%      35.7%   6.6pp
  2024  36.0%    31.0%      40.7%   9.7pp
  2025  36.4%    28.5%      39.6%  11.1pp


## 5. Variação Inter-Hospitalar: Quem Piorou vs Quem Melhorou

O efeito sistema é uniforme ou concentrado em hospitais específicos? Se poucos hospitais estão puxando a média para cima, intervenções direcionadas são possíveis.

In [6]:
# Hospital-level comparison: pre-COVID vs post-COVID late
hosp_pre = pre_covid.groupby("CNES").agg(
    n_pre=("MORTE", "count"), deaths_pre=("MORTE", "sum"), mort_pre=("MORTE", "mean")
).reset_index()

hosp_post = post_late.groupby("CNES").agg(
    n_post=("MORTE", "count"), deaths_post=("MORTE", "sum"), mort_post=("MORTE", "mean")
).reset_index()

hosp_compare = hosp_pre.merge(hosp_post, on="CNES", how="inner")
hosp_compare = hosp_compare[(hosp_compare["n_pre"] >= 30) & (hosp_compare["n_post"] >= 30)].copy()
hosp_compare["mort_change"] = hosp_compare["mort_post"] - hosp_compare["mort_pre"]
hosp_compare["mort_change_pp"] = hosp_compare["mort_change"] * 100
hosp_compare["total_n"] = hosp_compare["n_pre"] + hosp_compare["n_post"]

# Merge hospital metadata
hosp_compare = hosp_compare.merge(tags[["CNES", "icu_capacity_level", "broad_type", "nat_jur_category"]], on="CNES", how="left")

improved = hosp_compare[hosp_compare["mort_change"] < -0.02]
worsened = hosp_compare[hosp_compare["mort_change"] > 0.02]
stable = hosp_compare[(hosp_compare["mort_change"] >= -0.02) & (hosp_compare["mort_change"] <= 0.02)]

fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# Panel A: distribution of hospital-level mortality change
ax = axes[0]
ax.hist(hosp_compare["mort_change_pp"], bins=30, color=COLORS["primary"], alpha=0.7, edgecolor="white")
ax.axvline(0, color="black", linewidth=1)
ax.axvline(hosp_compare["mort_change_pp"].median(), color=COLORS["danger"], linewidth=2,
           linestyle="--", label=f"Mediana: {hosp_compare['mort_change_pp'].median():+.1f}pp")
ax.set_title(f"A. Distribuição da Variação de Mortalidade\n({len(hosp_compare)} hospitais com n≥30 em ambos os períodos)",
             fontweight="bold")
ax.set_xlabel("Variação (pp)")
ax.set_ylabel("Hospitais")
ax.legend()

# Panel B: by ICU capacity level
ax = axes[1]
for lvl in icu_levels:
    grp = hosp_compare[hosp_compare["icu_capacity_level"] == lvl]
    if len(grp) > 3:
        ax.boxplot(grp["mort_change_pp"].values, positions=[icu_levels.index(lvl)],
                   widths=0.5, patch_artist=True,
                   boxprops=dict(facecolor=icu_level_colors[lvl], alpha=0.6))
ax.set_xticks(range(len(icu_levels)))
ax.set_xticklabels([icu_level_labels.get(l, l) for l in icu_levels], fontsize=9)
ax.axhline(0, color="black", linewidth=0.5)
ax.set_title("B. Variação por Nível de UTI do Hospital", fontweight="bold")
ax.set_ylabel("Variação (pp)")

# Panel C: improved vs worsened vs stable
ax = axes[2]
categories = ["Melhorou\n(>2pp)", "Estável\n(±2pp)", "Piorou\n(>2pp)"]
counts = [len(improved), len(stable), len(worsened)]
extra_deaths = [
    int(improved["deaths_post"].sum() - (improved["n_post"] * improved["mort_pre"]).sum()),
    int(stable["deaths_post"].sum() - (stable["n_post"] * stable["mort_pre"]).sum()),
    int(worsened["deaths_post"].sum() - (worsened["n_post"] * worsened["mort_pre"]).sum()),
]
bar_colors_cat = [COLORS["success"], COLORS["muted"], COLORS["danger"]]
bars = ax.bar(categories, counts, color=bar_colors_cat, alpha=0.8)
for bar, cnt, ed in zip(bars, counts, extra_deaths):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f"{cnt} hosp.\n{ed:+,} óbitos", ha="center", fontsize=9, fontweight="bold")
ax.set_title("C. Hospitais que Pioraram vs Melhoraram", fontweight="bold")
ax.set_ylabel("Nº de Hospitais")

fig.suptitle("Variação Inter-Hospitalar: Pré-COVID → Pós-COVID Tardio",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
save_plot(fig, "hospital_variation", prefix="03")

print(f"\nHospitais analisados: {len(hosp_compare)} (n≥30 em ambos os períodos)")
print(f"  Melhoraram (>2pp): {len(improved)} ({len(improved)/len(hosp_compare):.0%})")
print(f"  Estáveis (±2pp):   {len(stable)} ({len(stable)/len(hosp_compare):.0%})")
print(f"  Pioraram (>2pp):   {len(worsened)} ({len(worsened)/len(hosp_compare):.0%})")
print(f"  Variação mediana:  {hosp_compare['mort_change_pp'].median():+.1f}pp")
print(f"\n  Excesso de óbitos (pós vs se mantivesse taxa pré):")
total_excess = int(hosp_compare["deaths_post"].sum() - (hosp_compare["n_post"] * hosp_compare["mort_pre"]).sum())
print(f"    Total: {total_excess:+,}")
print(f"    Nos hospitais que pioraram: {extra_deaths[2]:+,} ({extra_deaths[2]/total_excess:.0%} do total)")

  Saved: 03_hospital_variation.png

Hospitais analisados: 170 (n≥30 em ambos os períodos)
  Melhoraram (>2pp): 71 (42%)
  Estáveis (±2pp):   18 (11%)
  Pioraram (>2pp):   81 (48%)
  Variação mediana:  +1.2pp

  Excesso de óbitos (pós vs se mantivesse taxa pré):
    Total: +523
    Nos hospitais que pioraram: +1,572 (301% do total)


## 6. Top 10 Hospitais que Mais Pioraram vs Top 10 que Mais Melhoraram

Nominação dos hospitais para viabilizar ação.

In [7]:
from shared import load_cnes_names, hospital_name

names_df = load_cnes_names()
print(f"Loaded {len(names_df)} hospital names")

hosp_named = hosp_compare.copy()
hosp_named["nome_hosp"] = hosp_named["CNES"].apply(lambda c: hospital_name(c, names_df))

# Top 10 most worsened (weighted by volume)
hosp_named["excess_deaths"] = (hosp_named["deaths_post"] - hosp_named["n_post"] * hosp_named["mort_pre"]).astype(int)
top_worsened = hosp_named.nlargest(10, "excess_deaths")
top_improved = hosp_named.nsmallest(10, "excess_deaths")

print("\n" + "=" * 90)
print("TOP 10 HOSPITAIS COM MAIOR EXCESSO DE ÓBITOS (pré-COVID → pós-COVID tardio)")
print("=" * 90)
for _, h in top_worsened.iterrows():
    print(f"  {h['nome_hosp'][:45]:<45} | Pré: {h['mort_pre']:.1%} → Pós: {h['mort_post']:.1%} "
          f"({h['mort_change_pp']:+.1f}pp) | Excesso: {h['excess_deaths']:+,} óbitos "
          f"| UTI: {h['icu_capacity_level']}")

print("\n" + "=" * 90)
print("TOP 10 HOSPITAIS QUE MAIS MELHORARAM")
print("=" * 90)
for _, h in top_improved.iterrows():
    print(f"  {h['nome_hosp'][:45]:<45} | Pré: {h['mort_pre']:.1%} → Pós: {h['mort_post']:.1%} "
          f"({h['mort_change_pp']:+.1f}pp) | Excesso: {h['excess_deaths']:+,} óbitos "
          f"| UTI: {h['icu_capacity_level']}")

Loaded 486 hospital names

TOP 10 HOSPITAIS COM MAIOR EXCESSO DE ÓBITOS (pré-COVID → pós-COVID tardio)
  CENTRO HOSPITALAR DE SANTO ANDRE DR NEWTON DA | Pré: 25.5% → Pós: 51.9% (+26.5pp) | Excesso: +163 óbitos | UTI: small_icu
  COMPLEXO HOSPITALAR IRMA DULCE O S S          | Pré: 50.8% → Pós: 85.8% (+35.0pp) | Excesso: +64 óbitos | UTI: small_icu
  SANTA CASA DE SAO PAULO HOSPITAL CENTRAL SAO  | Pré: 18.0% → Pós: 39.6% (+21.6pp) | Excesso: +57 óbitos | UTI: small_icu
  HOSPITAL SANTA LYDIA RIBEIRAO PRETO           | Pré: 7.2% → Pós: 41.4% (+34.2pp) | Excesso: +53 óbitos | UTI: no_icu
  INSTITUTO DO CANCER DO ESTADO DE SAO PAULO    | Pré: 54.1% → Pós: 69.5% (+15.4pp) | Excesso: +52 óbitos | UTI: medium_icu
  SANTA CASA DE RIO CLARO                       | Pré: 5.1% → Pós: 24.5% (+19.4pp) | Excesso: +49 óbitos | UTI: no_icu
  HOSPITAL MUNICIPAL PIMENTAS BONSUCESSO MANUEL | Pré: 27.3% → Pós: 66.4% (+39.1pp) | Excesso: +45 óbitos | UTI: no_icu
  COMPLEXO HOSPITALAR PREFEITO EDIVALDO ORSI 

## 7. Mudança nos Subtipos e Procedimentos

Verificar se mudanças na codificação CID ou no mix de procedimentos explicam parte do efeito.

In [8]:
# Subtype analysis
resp["subtype_code"] = resp["DIAG_PRINC"].astype(str).str[:4]
subtype_labels = {"J960": "J96.0 Aguda", "J961": "J96.1 Crônica", "J969": "J96.9 NE"}
resp["subtype_label"] = resp["subtype_code"].map(subtype_labels).fillna("Outros")

main_subs = ["J96.0 Aguda", "J96.9 NE", "J96.1 Crônica"]
sub_yearly = resp.groupby(["year", "subtype_label"]).agg(
    n=("MORTE", "count"), mortality=("MORTE", "mean")
).reset_index()

# Procedure share by era
top_procs = resp["PROC_REA"].value_counts().head(8).index.tolist()
proc_era = []
for era in ERA_ORDER:
    era_d = resp[resp["covid_era"] == era]
    if len(era_d) == 0:
        continue
    for proc in top_procs:
        proc_d = era_d[era_d["PROC_REA"] == proc]
        proc_era.append({"era": era, "proc": proc, "share": len(proc_d)/len(era_d),
                         "mortality": proc_d["MORTE"].mean() if len(proc_d) > 0 else np.nan,
                         "n": len(proc_d)})

fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# Panel A: subtype volume trends
ax = axes[0]
sub_colors = [COLORS["danger"], COLORS["warning"], COLORS["primary"]]
for sub, color in zip(main_subs, sub_colors):
    grp = sub_yearly[sub_yearly["subtype_label"] == sub].sort_values("year")
    ax.plot(grp["year"], grp["n"], marker="o", linewidth=2, label=sub, color=color)
ax.set_title("A. Volume por Subtipo J96", fontweight="bold")
ax.set_ylabel("Internações")
ax.legend(fontsize=9)

# Panel B: subtype mortality trends
ax = axes[1]
for sub, color in zip(main_subs, sub_colors):
    grp = sub_yearly[sub_yearly["subtype_label"] == sub].sort_values("year")
    ax.plot(grp["year"], grp["mortality"] * 100, marker="o", linewidth=2, label=sub, color=color)
ax.set_title("B. Mortalidade por Subtipo", fontweight="bold")
ax.set_ylabel("Mortalidade (%)")
ax.legend(fontsize=9)

# Panel C: subtype share shift
ax = axes[2]
sub_share = resp.groupby(["year", "subtype_label"]).size().unstack(fill_value=0)
sub_share_pct = sub_share.div(sub_share.sum(axis=1), axis=0) * 100
for sub, color in zip(main_subs, sub_colors):
    if sub in sub_share_pct.columns:
        ax.plot(sub_share_pct.index, sub_share_pct[sub], marker="o", linewidth=2, label=sub, color=color)
ax.set_title("C. Proporção de Subtipos ao Longo do Tempo", fontweight="bold")
ax.set_ylabel("% do Total")
ax.legend(fontsize=9)

fig.suptitle("Evolução dos Subtipos J96 — Migração de Codificação?", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
save_plot(fig, "subtype_trends", prefix="03")

# Subtype share and mortality comparison
print("\nProporção e mortalidade dos subtipos:")
print(f"{'Subtipo':<20} {'2016 %':>8} {'2024 %':>8} {'Δ share':>8} | {'Mort 2016':>10} {'Mort 2024':>10} {'Δ mort':>8}")
for sub in main_subs:
    if sub in sub_share_pct.columns:
        s16 = sub_share_pct.loc[2016, sub] if 2016 in sub_share_pct.index else 0
        s24 = sub_share_pct.loc[2024, sub] if 2024 in sub_share_pct.index else 0
        m16 = sub_yearly[(sub_yearly["year"]==2016) & (sub_yearly["subtype_label"]==sub)]["mortality"].values
        m24 = sub_yearly[(sub_yearly["year"]==2024) & (sub_yearly["subtype_label"]==sub)]["mortality"].values
        m16v = m16[0] if len(m16) > 0 else np.nan
        m24v = m24[0] if len(m24) > 0 else np.nan
        print(f"  {sub:<18} {s16:>7.1f}% {s24:>7.1f}% {s24-s16:>+7.1f}pp | {m16v:>9.1%} {m24v:>9.1%} {(m24v-m16v)*100:>+7.1f}pp")

print("\nProcedimento dominante (0303140135) por era:")
for era in ERA_ORDER:
    row = [r for r in proc_era if r["era"] == era and r["proc"] == "0303140135"]
    if row:
        r = row[0]
        print(f"  {ERA_LABELS[era]:<35}: {r['share']:.1%} share, mort={r['mortality']:.1%} (n={r['n']:,})")

  Saved: 03_subtype_trends.png

Proporção e mortalidade dos subtipos:
Subtipo                2016 %   2024 %  Δ share |  Mort 2016  Mort 2024   Δ mort
  J96.0 Aguda           92.8%    76.9%   -15.9pp |     32.2%     38.2%    +6.0pp
  J96.9 NE               5.8%    16.1%   +10.3pp |     24.2%     39.2%   +15.0pp
  J96.1 Crônica          1.3%     5.0%    +3.7pp |     13.8%     12.4%    -1.4pp

Procedimento dominante (0303140135) por era:
  Pre-COVID (2016–2019)              : 80.9% share, mort=32.0% (n=35,948)
  COVID Acute (2020–2021)            : 80.7% share, mort=33.3% (n=21,255)
  Post-COVID Early (2022–H1 2023)    : 80.7% share, mort=34.5% (n=13,925)
  Post-COVID Late (H2 2023–2025)     : 83.4% share, mort=35.9% (n=23,641)


## 8. Comorbidades: O Perfil Mudou?

Comparação da prevalência de comorbidades entre eras, com teste de significância.

In [9]:
# Comorbidity prevalence and mortality impact by era
comorb_results = []
for col in comorbidity_cols:
    name = col.replace("comorbidity_", "").replace("_", " ").title()
    prev_pre = pre_covid[col].mean()
    prev_post = post_late[col].mean()
    mort_with = resp[resp[col] == 1]["MORTE"].mean()
    mort_without = resp[resp[col] == 0]["MORTE"].mean()
    # Chi-squared test for prevalence change
    c_pre = pre_covid[col].sum()
    c_post = post_late[col].sum()
    n_pre = len(pre_covid)
    n_post = len(post_late)
    if c_pre + c_post > 10:
        table = [[c_pre, n_pre - c_pre], [c_post, n_post - c_post]]
        chi2, p, _, _ = stats.chi2_contingency(table)
    else:
        chi2, p = 0, 1.0
    comorb_results.append({
        "comorbidity": name, "prev_pre": prev_pre, "prev_post": prev_post,
        "change_pp": (prev_post - prev_pre) * 100, "mort_with": mort_with,
        "mort_without": mort_without, "impact_pp": (mort_with - mort_without) * 100,
        "p": p
    })

comorb_df = pd.DataFrame(comorb_results).sort_values("impact_pp", ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

# Panel A: mortality impact (with vs without)
top_impact = comorb_df[comorb_df["prev_pre"] > 0.001].head(8)
y_pos = range(len(top_impact))
ax1.barh(y_pos, top_impact["impact_pp"], color=COLORS["danger"], alpha=0.8)
ax1.set_yticks(y_pos)
ax1.set_yticklabels(top_impact["comorbidity"].values)
ax1.set_xlabel("Impacto na Mortalidade (pp)")
ax1.set_title("A. Impacto na Mortalidade (com vs sem)", fontweight="bold")
ax1.invert_yaxis()
for i, (_, r) in enumerate(top_impact.iterrows()):
    ax1.text(r["impact_pp"] + 0.5, i, f"+{r['impact_pp']:.0f}pp", va="center", fontsize=9)

# Panel B: prevalence change (pre vs post)
sig_comorbs = comorb_df[comorb_df["p"] < 0.05].sort_values("change_pp")
if len(sig_comorbs) > 0:
    bar_colors_c = [COLORS["danger"] if c > 0 else COLORS["success"] for c in sig_comorbs["change_pp"]]
    ax2.barh(range(len(sig_comorbs)), sig_comorbs["change_pp"], color=bar_colors_c, alpha=0.8)
    ax2.set_yticks(range(len(sig_comorbs)))
    ax2.set_yticklabels(sig_comorbs["comorbidity"].values)
    ax2.axvline(0, color="black", linewidth=0.5)
    ax2.set_xlabel("Variação de Prevalência (pp)")
    ax2.set_title("B. Mudanças Significativas (p<0.05)\nPré-COVID → Pós-COVID Tardio", fontweight="bold")
    ax2.invert_yaxis()
else:
    ax2.text(0.5, 0.5, "Nenhuma comorbidade com\nmudança significativa", transform=ax2.transAxes,
             ha="center", va="center", fontsize=12)
    ax2.set_title("B. Mudanças de Prevalência", fontweight="bold")

fig.suptitle("Comorbidades: Impacto e Evolução", fontsize=14, fontweight="bold")
plt.tight_layout()
save_plot(fig, "comorbidity_analysis", prefix="03")

print("\nComorbidades — impacto e variação de prevalência:")
print(f"{'Comorbidade':<20} {'Impacto':>8} {'Prev Pré':>9} {'Prev Pós':>9} {'Δ Prev':>8} {'p-value':>8}")
for _, r in comorb_df.iterrows():
    sig = "***" if r['p'] < 0.001 else "**" if r['p'] < 0.01 else "*" if r['p'] < 0.05 else ""
    print(f"  {r['comorbidity']:<18} {r['impact_pp']:>+7.1f}pp {r['prev_pre']:>8.2%} {r['prev_post']:>8.2%} "
          f"{r['change_pp']:>+7.2f}pp {r['p']:>7.4f} {sig}")

  Saved: 03_comorbidity_analysis.png

Comorbidades — impacto e variação de prevalência:
Comorbidade           Impacto  Prev Pré  Prev Pós   Δ Prev  p-value
  Stroke               +38.1pp    1.03%    1.00%   -0.03pp  0.7552 
  Sepsis               +36.6pp    2.50%    2.78%   +0.28pp  0.0209 *
  Kidney Failure       +32.1pp    2.41%    3.02%   +0.61pp  0.0000 ***
  Pulmonary Fibrosis   +26.2pp    0.18%    0.21%   +0.03pp  0.4284 
  Heart Failure        +23.8pp    1.60%    1.81%   +0.20pp  0.0388 *
  Diabetes             +22.7pp    2.08%    2.20%   +0.12pp  0.2696 
  Ischemic Heart       +19.9pp    0.57%    0.45%   -0.12pp  0.0298 *
  Copd                 +17.8pp    2.79%    2.83%   +0.04pp  0.7559 
  Hypertension         +17.0pp    4.44%    4.47%   +0.03pp  0.8611 
  Covid                 +6.0pp    0.00%    0.69%   +0.69pp  0.0000 ***
  Obesity               +1.2pp    0.41%    0.32%   -0.09pp  0.0603 
  Asthma               -25.4pp    1.47%    2.26%   +0.79pp  0.0000 ***


## 9. Síntese: Triangulação dos Três Eixos

In [10]:
# Final metrics
row_2024 = decomp[decomp["year"] == 2024].iloc[0]
row_2025 = decomp[decomp["year"] == 2025].iloc[0]

metrics = {
    "baseline_mortality": round(baseline_overall, 4),
    "mortality_2024": round(row_2024["observed"], 4),
    "mortality_2025": round(row_2025["observed"], 4),
    "total_change_2024_pp": round(row_2024["total_change"] * 100, 1),
    "composition_effect_2024_pp": round(row_2024["composition_effect"] * 100, 1),
    "rate_effect_2024_pp": round(row_2024["rate_effect"] * 100, 1),
    "pct_rate_effect_2024": round(abs(row_2024["rate_effect"]) / (abs(row_2024["composition_effect"]) + abs(row_2024["rate_effect"])) * 100, 0),
    "total_change_2025_pp": round(row_2025["total_change"] * 100, 1),
    "composition_effect_2025_pp": round(row_2025["composition_effect"] * 100, 1),
    "rate_effect_2025_pp": round(row_2025["rate_effect"] * 100, 1),
    "mortality_gap_icu_2016_pp": round(icu_df[icu_df["year"]==2016]["gap"].values[0] * 100, 1),
    "mortality_gap_icu_2025_pp": round(icu_df[icu_df["year"]==2025]["gap"].values[0] * 100, 1),
    "hospitals_analyzed": len(hosp_compare),
    "hospitals_worsened": len(worsened),
    "hospitals_improved": len(improved),
    "hospitals_stable": len(stable),
    "median_hospital_change_pp": round(hosp_compare["mort_change_pp"].median(), 1),
    "total_excess_deaths_post_late": total_excess,
    "age_change_pre_covid": [
        {"age_group": r["age_group"], "pre": round(r["pre"], 4), "post": round(r["post"], 4),
         "change_pp": round(r["change_pp"], 1), "p": round(r["p"], 4)}
        for _, r in change_df.iterrows()
    ],
}

save_metrics(metrics, "03_mortality_drivers")

print("\n" + "=" * 70)
print("TRIANGULAÇÃO — TRÊS EIXOS INDEPENDENTES")
print("=" * 70)
print(f"\n1. DECOMPOSIÇÃO KITAGAWA (2024):")
print(f"   Total: {row_2024['total_change']*100:+.1f}pp")
print(f"   Composição (demografia): {row_2024['composition_effect']*100:+.1f}pp ({100-int(metrics['pct_rate_effect_2024'])}%)")
print(f"   Taxa (sistema): {row_2024['rate_effect']*100:+.1f}pp ({int(metrics['pct_rate_effect_2024'])}%)")
print(f"\n2. ANÁLISE DENTRO-DE-ESTRATO:")
sig_ages = change_df[change_df["p"] < 0.05]
print(f"   {len(sig_ages)} de {len(change_df)} faixas etárias com aumento significativo (p<0.05)")
print(f"   Aumento médio: {sig_ages['change_pp'].mean():+.1f}pp")
print(f"\n3. VARIAÇÃO INTER-HOSPITALAR:")
print(f"   {len(worsened)} hospitais pioraram >2pp ({len(worsened)/len(hosp_compare):.0%})")
print(f"   {len(improved)} hospitais melhoraram >2pp ({len(improved)/len(hosp_compare):.0%})")
print(f"   Gap UTI vs sem-UTI: {icu_df[icu_df['year']==2016]['gap'].values[0]*100:.1f}pp (2016) → {icu_df[icu_df['year']==2025]['gap'].values[0]*100:.1f}pp (2025)")
print(f"\nVEREDICTO: O aumento de mortalidade é predominantemente efeito SISTEMA,")
print(f"           concentrado em hospitais sem UTI e em pacientes sem acesso a UTI.")

  Saved metrics: 03_mortality_drivers.json

TRIANGULAÇÃO — TRÊS EIXOS INDEPENDENTES

1. DECOMPOSIÇÃO KITAGAWA (2024):
   Total: +6.1pp
   Composição (demografia): +2.2pp (36%)
   Taxa (sistema): +3.9pp (64%)

2. ANÁLISE DENTRO-DE-ESTRATO:
   5 de 6 faixas etárias com aumento significativo (p<0.05)
   Aumento médio: +3.5pp

3. VARIAÇÃO INTER-HOSPITALAR:
   81 hospitais pioraram >2pp (48%)
   71 hospitais melhoraram >2pp (42%)
   Gap UTI vs sem-UTI: 4.4pp (2016) → 11.1pp (2025)

VEREDICTO: O aumento de mortalidade é predominantemente efeito SISTEMA,
           concentrado em hospitais sem UTI e em pacientes sem acesso a UTI.
